In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder

# Load model
data = joblib.load("loan_project_final.pkl")
model = data['model']
feature_columns = data['feature_columns']
# Load test data
df = pd.read_csv("Test_new.csv")

# ==============================
# SAME PREPROCESSING AS TRAINING
# ==============================

# Fill missing values
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].median())

# Fix Dependents
df['Dependents'] = df['Dependents'].replace('3+', 3)
df['Dependents'] = df['Dependents'].astype(int)

# ==============================
# FEATURE ENGINEERING (SAME)
# ==============================
# Feature Engineering (EXACT MATCH)

df['ApplicantIncome_log'] = np.log(df['ApplicantIncome'])

df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

df['LoanIncomeRatio'] = df['LoanAmount'] / df['TotalIncome']

df['Total_Income_log'] = np.log(df['TotalIncome'])

df['LoanAmount_log'] = np.log(df['LoanAmount'])   # 🔥 MISSING ONE
# ==============================
# ENCODING (MATCH TRAINING STYLE)
# ==============================

le = LabelEncoder()

loan_col = ['Gender','Married','Education','Self_Employed','Property_Area','Dependents']

for col in loan_col:
    df[col] = le.fit_transform(df[col].astype(str))

# ==============================
# SPLIT
# ==============================
X = df.drop(['Loan_ID','Loan_Status'], axis=1)
y_actual = df['Loan_Status']

# Encode target also (same as training)
y_actual = le.fit_transform(y_actual)
X = X[feature_columns]
# ==============================
# PREDICT
# ==============================
y_pred = model.predict(X)

# ==============================
# COMPARE
# ==============================
result = pd.DataFrame({
    'Actual': y_actual,
    'Predicted': y_pred
})

result['Match'] = result['Actual'] == result['Predicted']

print(result.head(20))

# Save
result.to_csv("final_results.csv", index=False)
print("\nSaved → final_results.csv")

# Count correct predictions
correct = (y_actual == y_pred).sum()
total = len(y_actual)

print(f"Correct Predictions: {correct}/{total}")
print(f"Accuracy: {correct/total:.2f}")

    Actual  Predicted  Match
0        1          0  False
1        0          1  False
2        1          1   True
3        1          1   True
4        1          1   True
5        1          1   True
6        1          1   True
7        1          1   True
8        1          1   True
9        1          1   True
10       0          0   True
11       1          1   True
12       1          1   True
13       1          1   True
14       1          1   True
15       1          0  False
16       1          1   True
17       0          0   True
18       1          1   True
19       1          1   True

Saved → final_results.csv
Correct Predictions: 9611/10000
Accuracy: 0.96


In [7]:
model = joblib.load("loan_project_final.pkl")
print(type(model))
print(model)

<class 'dict'>
{'model': RandomForestClassifier(), 'scaler': StandardScaler(), 'label_encoders': LabelEncoder(), 'feature_columns': ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'ApplicantIncome_log', 'LoanAmount_log', 'TotalIncome', 'LoanIncomeRatio', 'Total_Income_log']}


In [8]:
import joblib

data = joblib.load("loan_project_final.pkl")

print(data.keys())

dict_keys(['model', 'scaler', 'label_encoders', 'feature_columns'])


In [10]:
from sklearn.metrics import classification_report

print(classification_report(y_actual, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.85      0.90      2038
           1       0.96      0.99      0.98      7962

    accuracy                           0.96     10000
   macro avg       0.96      0.92      0.94     10000
weighted avg       0.96      0.96      0.96     10000



In [11]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_actual, y_pred))

[[1735  303]
 [  86 7876]]
